# 08 — Presentation Figure Extraction
Generate and save all presentation-quality PNG figures to `output/presentation/`.
Run all cells top-to-bottom. Figures are numbered to match slide order in notebook 09.


In [2]:
## ── 0. Setup & Imports ──────────────────────────────────────────────────────
import os, sys, json
from pathlib import Path
from collections import OrderedDict

import polars as pl
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless – saves files without displaying
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns
import joblib
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────────────────────────────
project_root = Path.cwd().parent
sys.path.append(str(project_root / 'src'))

import data_processing
from config import DATA_PATH
from features.feature_engineering import CarPriceFeatureEngineer

# ── Output directory ──────────────────────────────────────────────────────────
OUT = project_root / 'output' / 'presentation'
OUT.mkdir(parents=True, exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
})
DARK_BLUE  = '#1B3A6B'
MID_BLUE   = '#2E6DA4'
LIGHT_GREY = '#F4F6F8'

def save(fig, name):
    path = OUT / name
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  ✓ saved  {name}')

print(f'Project root : {project_root}')
print(f'Output dir   : {OUT}')
print('✓ Setup complete')


Project root : /Users/brunobrumbrum/car_price_prediction
Output dir   : /Users/brunobrumbrum/car_price_prediction/output/presentation
✓ Setup complete


In [4]:
## ── 1. Load Training Data, Models & Metadata ────────────────────────────────
MODEL_DIR = project_root / 'models' / 'lean_quantile'

# ── metadata ──────────────────────────────────────────────────────────────────
with open(MODEL_DIR / 'metadata.json') as f:
    meta = json.load(f)

training_feature_cols = meta['features']['all_features']
n_ohe     = meta.get('n_brand_ohe', 12)
total_feats = len(training_feature_cols) + n_ohe
print(f'Features (numeric)  : {len(training_feature_cols)}')
print(f'Features (+ OHE)    : {total_feats}')
print(f'MAE Q50 (log)       : {meta["validation_mae_q50"]:.5f}')
print(f'Coverage Q15–Q85    : {meta["validation_coverage"]:.2f}%')

# ── models ────────────────────────────────────────────────────────────────────
lgb_q15 = joblib.load(MODEL_DIR / 'lgb_q15_lean.pkl')
lgb_q50 = joblib.load(MODEL_DIR / 'lgb_q50_lean.pkl')
lgb_q85 = joblib.load(MODEL_DIR / 'lgb_q85_lean.pkl')
feat_eng = joblib.load(MODEL_DIR / 'feature_engineer_lean.pkl')
print('✓ Models loaded')

# ── raw data ──────────────────────────────────────────────────────────────────
data_dir = DATA_PATH / 'le_boncoin_13_oct_2025'
print('Loading raw data …')
df_raw   = data_processing.load_car_data(data_dir)
print(f'  Raw rows: {len(df_raw):,}')
df_clean = data_processing.clean_car_data(df_raw)
print(f'  Clean rows: {len(df_clean):,}')
print('✓ Data loaded')


Features (numeric)  : 31
Features (+ OHE)    : 43
MAE Q50 (log)       : 0.18505
Coverage Q15–Q85    : 69.81%
✓ Models loaded
Loading raw data …
📊 Parsing horsepower from puissance_din column...
✅ Loaded 732,427 rows with horsepower parsed
   Note: 'energie' column contains fuel type (kept as-is)
(732427, 36)
  Raw rows: 732,427
🧹 Starting data cleaning pipeline...

1️⃣ Converting data types and normalizing text...
   Original: 732,427 rows
   After conversion: 732,426 rows
   Removed (invalid price): 1
   Unique brands: 145, Unique models: 1646

2️⃣ Removing antique cars (pre-1990)...
   Removed 14,536 antique cars

3️⃣ Removing 'autre' entries...
   Removed 4,347 'autre' entries

4️⃣ Cleaning horsepower...
   HP cleaning: dropped 5620 cars <50HP, 114 cars >1000HP, 35437 outliers (IQR per brand), 0 missing HP
   Remaining dataset - Mean HP: 139.7, Median HP: 125.0

5️⃣ Dropping rare brands (<400 cars)...
   Dropped 3,805 cars from 65 rare brands (< 400 observations)
   Remaining: 43 br

In [5]:
## ── FIGURE 01 — Top Brands by Listing Count ─────────────────────────────────
brand_counts = (
    df_clean
    .group_by('brand')
    .agg(pl.len().alias('count'))
    .sort('count', descending=True)
    .head(20)
    .to_pandas()
    .sort_values('count')
)

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(brand_counts['brand'], brand_counts['count'] / 1000, color=MID_BLUE)
ax.bar_label(bars, fmt='%.0fk', padding=3, fontsize=9)
ax.set_xlabel('Number of listings (thousands)', labelpad=8)
ax.set_title('Top 20 Brands — Training Data (620 K listings)', fontsize=13,
             fontweight='bold', color=DARK_BLUE, pad=12)
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor(LIGHT_GREY)
fig.patch.set_facecolor('white')
plt.tight_layout()
save(fig, '01_top_brands.png')


  ✓ saved  01_top_brands.png


In [6]:
## ── FIGURE 02 — Data Pipeline Funnel ────────────────────────────────────────
stages = [
    ('Le Boncoin scrape\n(Oct 2025)', 732_427, 146, 1_661),
    ('Remove pre-1990\n& "autre" brand', 690_000, 130, 1_600),
    ('HP outlier removal\n(IQR filter)', 680_000, 130, 1_600),
    ('Brand minimum\n(≥ 400 listings)', 650_000, 43, 1_200),
    ('Per-brand IQR price\n& km (×1.5)', 620_918, 43, 805),
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, len(stages) + 0.8)
ax.axis('off')
fig.patch.set_facecolor('white')

colors = ['#2E6DA4', '#3579B1', '#3D87BF', '#4495CD', '#4AA4DB']
arrow_props = dict(arrowstyle='->', color='#666666', lw=1.5)

for i, (label, rows, brands, models) in enumerate(stages):
    y    = len(stages) - i - 0.2
    width = 9 - i * 1.4
    x0   = (10 - width) / 2
    rect = mpatches.FancyBboxPatch(
        (x0, y - 0.35), width, 0.7,
        boxstyle='round,pad=0.05',
        linewidth=1, edgecolor='white',
        facecolor=colors[i], zorder=3
    )
    ax.add_patch(rect)
    ax.text(5, y, label, ha='center', va='center', fontsize=8.5,
            color='white', fontweight='bold', zorder=4, linespacing=1.2)
    ax.text(x0 - 0.15, y,
            f'{rows:,} rows\n{brands} brands\n{models} models',
            ha='right', va='center', fontsize=7.5, color='#333333', linespacing=1.3)
    if i < len(stages) - 1:
        ax.annotate('', xy=(5, y - 0.36), xytext=(5, y - 0.7),
                    arrowprops=arrow_props, zorder=5)

ax.set_title('Data Cleaning Pipeline', fontsize=14, fontweight='bold',
             color=DARK_BLUE, pad=8)
plt.tight_layout()
save(fig, '02_data_pipeline.png')


  ✓ saved  02_data_pipeline.png


In [7]:
## ── FIGURE 03 — Feature Groups Table ───────────────────────────────────────
groups = [
    ('Time',         6,  'car_age, decade, is_almost_new, sqrt_age, age², age³'),
    ('Brand stats', 12,  'median/mean/std/count/min/max/p25/p75/p90/iqr log-price + 2'),
    ('Model stats', 12,  'same 12 statistics at model level'),
    ('Brand OHE',   12,  'binary indicator for each of the 12 top brands'),
    ('Model size',   1,  'model_count / brand_count  (relative prevalence)'),
    ('TOTAL',       43,  ''),
]
col_labels = ['Feature Group', 'Count', 'Variables / Description']
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axis('off')
fig.patch.set_facecolor('white')

table_data = [[g, str(c), desc] for g, c, desc in groups]
tbl = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='left',
    loc='center',
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.auto_set_column_width([0, 1, 2])

# Style header
for j in range(3):
    tbl[0, j].set_facecolor(DARK_BLUE)
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Style TOTAL row
for j in range(3):
    tbl[len(groups), j].set_facecolor('#D0DCF0')
    tbl[len(groups), j].set_text_props(fontweight='bold')

# Alternate row shading
for i in range(1, len(groups)):
    bg = LIGHT_GREY if i % 2 == 0 else 'white'
    for j in range(3):
        tbl[i, j].set_facecolor(bg)

ax.set_title('Model Feature Space — 43 Input Columns', fontsize=13,
             fontweight='bold', color=DARK_BLUE, pad=12)
plt.tight_layout()
save(fig, '03_feature_groups.png')


  ✓ saved  03_feature_groups.png


In [8]:
## ── FIGURE 04 — Quantile Corridor Concept ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
fig.patch.set_facecolor('white')

# Example car "column" with quantile dots
def draw_car_col(ax, cx, q15_y, q50_y, q85_y, label, price_label):
    # Shaded band Q15–Q85
    band = mpatches.FancyBboxPatch(
        (cx - 0.35, q15_y), 0.7, q85_y - q15_y,
        boxstyle='round,pad=0.05',
        facecolor='#B3D1F0', edgecolor='#2E6DA4', linewidth=1.2, zorder=2
    )
    ax.add_patch(band)
    # Q50 line
    ax.plot([cx - 0.4, cx + 0.4], [q50_y, q50_y],
            color=DARK_BLUE, lw=2.5, zorder=3)
    # Labels
    ax.text(cx, q85_y + 0.18, f'Q85', ha='center', fontsize=8, color='#555')
    ax.text(cx, q50_y + 0.12, f'Q50', ha='center', fontsize=8,
            color=DARK_BLUE, fontweight='bold')
    ax.text(cx, q15_y - 0.22, f'Q15', ha='center', fontsize=8, color='#555')
    ax.text(cx, -0.3,  label,        ha='center', fontsize=9, color='#333',
            fontweight='bold')
    ax.text(cx, -0.55, price_label,  ha='center', fontsize=8, color='#666')

examples = [
    (1.5,  1.2, 2.0, 2.8,  'Renault Clio\n2018, 50 000 km',  '≈ €7 200'),
    (3.5,  2.0, 2.9, 4.0,  'Peugeot 308\n2019, 60 000 km',   '≈ €11 500'),
    (5.5,  2.8, 3.8, 4.8,  'BMW Série 3\n2017, 80 000 km',   '≈ €17 000'),
    (7.5,  3.5, 4.6, 5.4,  'Mercedes C\n2020, 40 000 km',    '≈ €25 000'),
]
for args in examples:
    draw_car_col(ax, *args)

# ── axis label ──
ax.annotate('', xy=(9.5, 1.0), xytext=(0.2, 1.0),
            arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.2))
ax.text(9.6, 1.0, '€', ha='left', va='center', fontsize=11, color='#aaa')

# Legend
band_patch  = mpatches.Patch(facecolor='#B3D1F0', edgecolor='#2E6DA4', label='70% Price Corridor (Q15–Q85)')
median_line = mlines.Line2D([], [], color=DARK_BLUE, lw=2.5, label='Median estimate (Q50)')
ax.legend(handles=[band_patch, median_line], loc='upper left', fontsize=9, framealpha=0.85)

ax.set_title('Quantile Regression — 70% Price Corridor per Listing',
             fontsize=13, fontweight='bold', color=DARK_BLUE, pad=10, y=1.01)
plt.tight_layout()
save(fig, '04_quantile_corridor.png')


  ✓ saved  04_quantile_corridor.png


In [10]:
## ── 2. Re-split Test Set & Generate Predictions ─────────────────────────────
# Feature engineering — pass Polars DataFrame directly
X_all_pl = feat_eng.transform(df_clean.select(['brand', 'model', 'year', 'km']))
X_all    = X_all_pl.to_pandas()
y_all    = np.log(df_clean['price'].to_numpy())

# Align to training columns (add missing OHE columns as 0)
for col in training_feature_cols:
    if col not in X_all.columns:
        X_all[col] = 0
for col in X_all.columns:
    if X_all[col].dtype == 'object':
        X_all[col] = pd.to_numeric(X_all[col], errors='coerce').fillna(0)
X_all = X_all[training_feature_cols]

# Train/test split identical to training
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42
)

# Predictions on test set
pred_q15_log = lgb_q15.predict(X_test)
pred_q50_log = lgb_q50.predict(X_test)
pred_q85_log = lgb_q85.predict(X_test)

pred_q15 = np.exp(pred_q15_log)
pred_q50 = np.exp(pred_q50_log)
pred_q85 = np.exp(pred_q85_log)
actual   = np.exp(y_test)

# Metrics
mae_eur  = mean_absolute_error(actual, pred_q50)
mape     = np.mean(np.abs((actual - pred_q50) / actual)) * 100
r2       = r2_score(y_test, pred_q50_log)         # on log scale
coverage = np.mean((actual >= pred_q15) & (actual <= pred_q85)) * 100

print(f'MAE  (€)     : {mae_eur:,.0f}')
print(f'MAPE (%)     : {mape:.1f}%')
print(f'R²   (log)   : {r2:.4f}')
print(f'Coverage (%) : {coverage:.1f}%')
print('✓ Test predictions computed')


MAE  (€)     : 2,536
MAPE (%)     : 19.3%
R²   (log)   : 0.8967
Coverage (%) : 70.1%
✓ Test predictions computed


In [11]:
## ── FIGURE 05 — Metrics Summary Table ──────────────────────────────────────
n_train = len(X_train)
n_test  = len(X_test)

rows = [
    ['Metric',              'Value'],
    ['MAE (median price)',  f'€ {mae_eur:,.0f}'],
    ['MAPE',                f'{mape:.1f} %'],
    ['R² (log scale)',      f'{r2:.4f}'],
    ['Coverage Q15–Q85',    f'{coverage:.1f} %'],
    ['Training samples',    f'{n_train:,}'],
    ['Test samples',        f'{n_test:,}'],
    ['n_estimators',        '5 000'],
    ['Learning rate',       '0.10'],
    ['Quantile targets',    'α = 0.15 / 0.50 / 0.85'],
]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.axis('off')
fig.patch.set_facecolor('white')

tbl = ax.table(
    cellText=rows[1:],
    colLabels=rows[0],
    cellLoc='left',
    loc='center',
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.auto_set_column_width([0, 1])

for j in range(2):
    tbl[0, j].set_facecolor(DARK_BLUE)
    tbl[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, len(rows)):
    bg = LIGHT_GREY if i % 2 == 0 else 'white'
    for j in range(2):
        tbl[i, j].set_facecolor(bg)
        tbl[i, j].set_height(0.10)

ax.set_title('LightGBM Quantile Model — Performance Summary',
             fontsize=13, fontweight='bold', color=DARK_BLUE, pad=12)
plt.tight_layout()
save(fig, '05_metrics_table.png')


  ✓ saved  05_metrics_table.png


In [ ]:
## ── FIGURE 06 — Actual vs Predicted Scatter ────────────────────────────────
rng   = np.random.default_rng(42)
idx   = rng.choice(len(actual), size=min(2000, len(actual)), replace=False)

act_s  = actual[idx]
p50_s  = pred_q50[idx]
p15_s  = pred_q15[idx]
p85_s  = pred_q85[idx]

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor('white')

# Error bars = Q15–Q85 interval
yerr = np.array([p50_s - p15_s, p85_s - p50_s])
ax.errorbar(act_s / 1000, p50_s / 1000,
            yerr=yerr / 1000,
            fmt='o', alpha=0.25, ms=3, lw=0.6,
            color=MID_BLUE, ecolor='#90B8D8', zorder=2,
            label='Median prediction ± Q15–Q85')

# Perfect prediction diagonal
lim = max(act_s.max(), p50_s.max()) / 1000 * 1.02
ax.plot([0, lim], [0, lim], '--', color='#CC3333', lw=1.5, label='Perfect prediction', zorder=3)

ax.set_xlabel('Actual Price (k€)',   fontsize=11)
ax.set_ylabel('Predicted Q50 (k€)', fontsize=11)
ax.set_title('Actual vs Predicted — Test Set (2 000 sample)',
             fontsize=13, fontweight='bold', color=DARK_BLUE)
ax.legend(fontsize=9)
ax.set_facecolor(LIGHT_GREY)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save(fig, '06_actual_vs_predicted.png')


AttributeError: 'numpy.ndarray' object has no attribute 'values'

In [ ]:
## ── FIGURE 07 — Residuals Distribution ─────────────────────────────────────
residuals_log = pred_q50_log - y_test          # predicted - actual (log scale)

fig, ax = plt.subplots(figsize=(8, 4.5))
fig.patch.set_facecolor('white')

ax.hist(residuals_log, bins=80, density=True, color=MID_BLUE,
        edgecolor='white', linewidth=0.3, alpha=0.8)

from scipy.stats import gaussian_kde
kde_x = np.linspace(residuals_log.min(), residuals_log.max(), 300)
kde   = gaussian_kde(residuals_log, bw_method=0.15)
ax.plot(kde_x, kde(kde_x), color=DARK_BLUE, lw=2, label='KDE')

ax.axvline(0, color='#CC3333', lw=1.8, linestyle='--', label='Zero residual')
ax.axvline(residuals_log.mean(), color='#E67E22', lw=1.4, linestyle=':',
           label=f'Mean = {residuals_log.mean():.3f}')

ax.set_xlabel('Residual  log(predicted Q50) − log(actual)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Residual Distribution — Test Set',
             fontsize=13, fontweight='bold', color=DARK_BLUE)
ax.legend(fontsize=9)
ax.set_facecolor(LIGHT_GREY)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save(fig, '07_residuals.png')


In [ ]:
## ── 3. SHAP Values — Q50 Model ───────────────────────────────────────────────
# Subsample 50 K from the test set for SHAP speed
shap_n   = min(50_000, len(X_test))
shap_idx = np.random.default_rng(42).choice(len(X_test), size=shap_n, replace=False)
X_shap   = X_test.iloc[shap_idx]

print(f'Computing SHAP values for {shap_n:,} samples …')
explainer   = shap.TreeExplainer(lgb_q50)
shap_values = explainer.shap_values(X_shap)
print(f'✓ SHAP values shape: {shap_values.shape}')


In [ ]:
## ── FIGURE 08 — SHAP Top-20 Dot Plot ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('white')

shap.summary_plot(
    shap_values, X_shap,
    max_display=20,
    plot_type='dot',
    show=False,
    plot_size=None,
    color_bar=True,
)
ax = plt.gca()
ax.set_title('SHAP Feature Importance — Q50 Model (Top 20)',
             fontsize=13, fontweight='bold', color=DARK_BLUE, pad=10)
fig = plt.gcf()
fig.patch.set_facecolor('white')
save(fig, '08_shap_top20.png')


In [ ]:
## ── 4. Load Customs Data (CI & CM) ──────────────────────────────────────────
# ── Model reference lists ─────────────────────────────────────────────────────
all_brands = list(feat_eng.brand_price_stats_.keys())
all_models = list(feat_eng.model_price_stats_.keys())

# ── Côte d'Ivoire ─────────────────────────────────────────────────────────────
ci_path = '/Users/brunobrumbrum/Downloads/VEHICULE_CHASSIS (1).xlsx'
df_ci_raw = pl.read_excel(ci_path)

df_ci = (
    df_ci_raw
    .with_columns([
        pl.col('MARQUE').str.to_lowercase().str.strip_chars()
          .str.replace('land rover', 'land-rover', literal=True).alias('brand'),
        pl.col('MODELE').str.to_lowercase().str.strip_chars().alias('model'),
        pl.col('PREMIERE_MIS_CIRCULAT').str.slice(0, 4).cast(int).alias('year'),
        (pl.col('VALCAF') * 0.0015).alias('actual_price_eur'),
        pl.lit(100_000).alias('km'),
        pl.lit('CI').alias('country'),
    ])
    .with_columns([
        pl.col('brand').replace({'land-rover': 'landrover', 'mercedes-benz': 'mercedesbenz'}).alias('brand'),
        pl.col('model').replace({'cr-v':'crv','rogue':'xtrail','mirage':'space star',
                                  'glc':'classe glc','tacoma':'hilux'}).alias('model'),
    ])
)
df_ci_clean = df_ci.filter(pl.col('brand').is_in(all_brands) & pl.col('model').is_in(all_models))
print(f'CI  raw: {len(df_ci):,}  →  filtered: {len(df_ci_clean):,}')

# ── Cameroon ──────────────────────────────────────────────────────────────────
cm_path = '/Users/brunobrumbrum/Downloads/Extraction_Jeu_données_Test_Vehicules (2).xlsx'
df_cm_raw = pl.read_excel(cm_path)

df_cm = (
    df_cm_raw
    .with_columns([
        pl.col('Marque').str.to_lowercase().str.strip_chars().alias('brand'),
        pl.col('Modèle').str.to_lowercase().str.strip_chars().alias('model'),
        pl.col('Année Fabrication').cast(pl.Int64).alias('year'),
        (pl.col('Valeur imposable (XAF)') * 0.0015).alias('actual_price_eur'),
        pl.lit(100_000).alias('km'),
        pl.lit('CM').alias('country'),
    ])
    .with_columns([
        pl.col('brand').replace({'land rover': 'landrover'}).alias('brand'),
        pl.col('model').replace({'rav 4':'rav4','toyota corolla':'corolla',
                                  'toyota yaris verso':'yaris verso',
                                  'c-max':'cmax','carina':'avensis'}).alias('model'),
    ])
)
df_cm_clean = df_cm.filter(pl.col('brand').is_in(all_brands) & pl.col('model').is_in(all_models))
print(f'CM  raw: {len(df_cm):,}  →  filtered: {len(df_cm_clean):,}')
print('✓ Customs data loaded')


In [ ]:
## ── Feature engineering & predictions for customs data ──────────────────────
def fe_predict(df_customs):
    """Apply feature engineering + predictions. Returns polars df with pred cols."""
    df_feat = feat_eng.transform(df_customs.select(['brand', 'model', 'year', 'km']))
    X = df_feat.to_pandas()
    for col in training_feature_cols:
        if col not in X.columns:
            X[col] = 0
    X = X[training_feature_cols]
    for col in X.columns:
        if X[col].dtype == 'object':
            X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
    q15 = np.exp(lgb_q15.predict(X))
    q50 = np.exp(lgb_q50.predict(X))
    q85 = np.exp(lgb_q85.predict(X))
    return df_customs.with_columns([
        pl.Series('pred_q15', q15),
        pl.Series('pred_q50', q50),
        pl.Series('pred_q85', q85),
    ])

df_ci_results = fe_predict(df_ci_clean)
df_cm_results = fe_predict(df_cm_clean)
print(f'CI predictions: {len(df_ci_results):,}')
print(f'CM predictions: {len(df_cm_results):,}')
print('✓ Customs predictions complete')


In [ ]:
## ── FIGURE 09 — KDE Year Distribution ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.patch.set_facecolor('white')

train_years = df_clean['year'].to_numpy()
ci_years    = df_ci_clean['year'].to_numpy()
cm_years    = df_cm_clean['year'].to_numpy()

sns.kdeplot(train_years, ax=ax, label=f'Training data (n={len(train_years):,})',
            color=MID_BLUE, linewidth=2)
sns.kdeplot(ci_years, ax=ax, label=f"Côte d'Ivoire (n={len(ci_years):,})",
            color='#27AE60', linewidth=2)
sns.kdeplot(cm_years, ax=ax, label=f'Cameroon (n={len(cm_years):,})',
            color='#E67E22', linewidth=2)

ax.set_xlabel('Vehicle Year', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Year Distribution — Training vs Customs Datasets',
             fontsize=13, fontweight='bold', color=DARK_BLUE)
ax.legend(fontsize=9)
ax.set_facecolor(LIGHT_GREY)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save(fig, '09_year_distribution.png')


In [ ]:
## ── FIGURE 10 — Customs Descriptive Stats Table ─────────────────────────────
ci_prices = df_ci_clean['actual_price_eur'].to_numpy()
cm_prices = df_cm_clean['actual_price_eur'].to_numpy()

stats_rows = [
    ['Metric',                      "Côte d'Ivoire",              'Cameroon'],
    ['Vehicles (after filter)',      f'{len(ci_prices):,}',        f'{len(cm_prices):,}'],
    ['Year range',
        f"{int(df_ci_clean['year'].min())}–{int(df_ci_clean['year'].max())}",
        f"{int(df_cm_clean['year'].min())}–{int(df_cm_clean['year'].max())}"],
    ['Median year',
        f"{df_ci_clean['year'].median():.0f}",
        f"{df_cm_clean['year'].median():.0f}"],
    ['Declared price — median',      f'€ {np.median(ci_prices):,.0f}',  f'€ {np.median(cm_prices):,.0f}'],
    ['Declared price — mean',        f'€ {np.mean(ci_prices):,.0f}',    f'€ {np.mean(cm_prices):,.0f}'],
    ['Declared price — std',         f'€ {np.std(ci_prices):,.0f}',     f'€ {np.std(cm_prices):,.0f}'],
    ['Declared price — min',         f'€ {np.min(ci_prices):,.0f}',     f'€ {np.min(cm_prices):,.0f}'],
    ['Declared price — max',         f'€ {np.max(ci_prices):,.0f}',     f'€ {np.max(cm_prices):,.0f}'],
    ['Unique brands',
        str(df_ci_clean['brand'].n_unique()),
        str(df_cm_clean['brand'].n_unique())],
    ['Unique models',
        str(df_ci_clean['model'].n_unique()),
        str(df_cm_clean['model'].n_unique())],
]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axis('off')
fig.patch.set_facecolor('white')

tbl = ax.table(
    cellText=stats_rows[1:],
    colLabels=stats_rows[0],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10.5)
tbl.auto_set_column_width([0, 1, 2])

for j in range(3):
    tbl[0, j].set_facecolor(DARK_BLUE)
    tbl[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, len(stats_rows)):
    bg = LIGHT_GREY if i % 2 == 0 else 'white'
    for j in range(3):
        tbl[i, j].set_facecolor(bg)
        tbl[i, j].set_height(0.09)

ax.set_title('Customs Data — Descriptive Statistics',
             fontsize=13, fontweight='bold', color=DARK_BLUE, pad=12)
plt.tight_layout()
save(fig, '10_customs_stats.png')


In [ ]:
## ── 5. Validation Data (reuse nb07 helpers) ─────────────────────────────────
# ── Inline copies of the three helpers from notebook 07 ──────────────────────
import matplotlib.lines as mlines

WARN_SUFFIX = ' [!]'


def prepare_validation_data(df_customs, df_training, country_code, year_flex=3):
    results = []
    df_train_pd = df_training.to_pandas()
    for row in df_customs.iter_rows(named=True):
        brand          = row['brand']
        model          = row['model']
        year           = row['year']
        declared_price = row['actual_price_eur']
        pred_q15_v     = row['pred_q15']
        pred_q50_v     = row['pred_q50']
        pred_q85_v     = row['pred_q85']
        mask = (
            (df_train_pd['brand'] == brand) &
            (df_train_pd['model'] == model) &
            (df_train_pd['year'] >= year - year_flex) &
            (df_train_pd['year'] <= year + year_flex)
        )
        matched = df_train_pd[mask]
        n = len(matched)
        if n > 0:
            prices = matched['price'].values
            train_min = float(np.min(prices));  train_max = float(np.max(prices))
            train_q15 = float(np.percentile(prices, 15))
            train_q50 = float(np.percentile(prices, 50))
            train_q85 = float(np.percentile(prices, 85))
            has_td = True
        else:
            train_min = train_q15 = train_q50 = train_q85 = train_max = np.nan
            has_td = False
        gap_pct = (pred_q50_v - declared_price) / pred_q50_v * 100
        if pred_q15_v <= declared_price <= pred_q85_v:
            tier = 'GREEN'
        elif has_td and (train_min <= declared_price <= train_max):
            tier = 'ORANGE'
        else:
            tier = 'RED'
        results.append({
            'brand': brand, 'model': model, 'year': year,
            'declared_price': declared_price,
            'pred_q15': pred_q15_v, 'pred_q50': pred_q50_v, 'pred_q85': pred_q85_v,
            'train_min': train_min, 'train_q15': train_q15, 'train_q50': train_q50,
            'train_q85': train_q85, 'train_max': train_max,
            'gap_pct': gap_pct, 'color_tier': tier,
            'training_sample_size': n, 'has_training_data': has_td,
        })
    df_r = pd.DataFrame(results).sort_values(['brand', 'model', 'year']).reset_index(drop=True)
    return df_r


def create_validation_plot(df_subset, title, median_declared, year_flex=3):
    n_cars = len(df_subset)
    if n_cars == 0:
        return None
    fig, ax = plt.subplots(figsize=(16, max(6, n_cars * 0.45)), dpi=150)
    color_map = {'GREEN': 'green', 'ORANGE': 'orange', 'RED': 'red'}
    labels = [
        f"{r['brand']} {r['model']} {int(r['year'])}" + (WARN_SUFFIX if r['training_sample_size'] < 5 else '')
        for _, r in df_subset.iterrows()
    ]
    y_positions = np.arange(n_cars)
    for i, (_, row) in enumerate(df_subset.iterrows()):
        y = y_positions[i]
        if row['has_training_data'] and not np.isnan(row['train_min']):
            ax.fill_betweenx([y-0.35, y+0.35], row['train_min'], row['train_max'],
                             color='lightgrey', alpha=0.35, zorder=1)
            ax.hlines(y, row['train_min'], row['train_max'], colors='#cccccc', lw=0.5, zorder=1)
            ax.vlines(row['train_q15'], y-0.15, y+0.15, color='#888888', lw=1.5, zorder=2)
            ax.vlines(row['train_q85'], y-0.15, y+0.15, color='#888888', lw=1.5, zorder=2)
            ax.vlines(row['train_q50'], y-0.30, y+0.30, color='#444444', lw=2.0, zorder=2)
        ax.hlines(y, row['pred_q15'], row['pred_q85'], colors='steelblue', lw=3, alpha=0.5, zorder=3)
        ax.vlines(row['pred_q15'], y-0.20, y+0.20, color='steelblue', lw=2.0, zorder=4)
        ax.vlines(row['pred_q85'], y-0.20, y+0.20, color='steelblue', lw=2.0, zorder=4)
        ax.vlines(row['pred_q50'], y-0.25, y+0.25, color='navy',      lw=2.5, zorder=4)
        ax.scatter(row['declared_price'], y, color=color_map[row['color_tier']],
                   s=60, zorder=5, edgecolors='black', linewidths=0.5)
    ax.set_xlim(auto=True)
    x_min, x_max = ax.get_xlim()
    x_range = x_max - x_min
    for i, (_, row) in enumerate(df_subset.iterrows()):
        gs = '+' if row['gap_pct'] > 0 else ''
        ax.annotate(f"{gs}{row['gap_pct']:.1f}%",
                    xy=(x_max - x_range*0.02, y_positions[i]),
                    fontsize=10, color=color_map[row['color_tier']], va='center', ha='right')
    ax.axvline(x=median_declared, color='purple', linestyle='--', lw=1.5, alpha=0.7, zorder=0)
    ax.annotate(f'Median declared\n€{median_declared:,.0f}',
                xy=(median_declared, n_cars - 0.5),
                fontsize=10, color='purple', ha='center', va='bottom')
    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel('Price (€)', fontsize=12)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x:,.0f}'))
    ax.tick_params(axis='x', labeltop=True, labelbottom=True)
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(True, axis='x', alpha=0.3, zorder=0)
    ax.invert_yaxis()
    fig.patch.set_facecolor('white')
    plt.tight_layout()
    return fig


print('✓ Validation helpers defined')


In [ ]:
## ── Prepare validation DataFrames ───────────────────────────────────────────
print('Preparing validation data CI …')
df_validation_ci = prepare_validation_data(df_ci_results, df_clean, 'CI', year_flex=3)
print(f'  CI: {len(df_validation_ci)} vehicles')

print('Preparing validation data CM …')
df_validation_cm = prepare_validation_data(df_cm_results, df_clean, 'CM', year_flex=3)
print(f'  CM: {len(df_validation_cm)} vehicles')
print('✓ Validation data ready')


In [ ]:
## ── FIGURE 11 — GREEN / ORANGE / RED Tier Bar Chart ────────────────────────
def tier_counts(df_val):
    c = df_val['color_tier'].value_counts()
    return {t: int(c.get(t, 0)) for t in ['GREEN', 'ORANGE', 'RED']}

ci_tiers = tier_counts(df_validation_ci)
cm_tiers = tier_counts(df_validation_cm)
ci_total = sum(ci_tiers.values())
cm_total = sum(cm_tiers.values())

x = np.arange(2)
labels_x = [f"Côte d'Ivoire\n(n={ci_total})", f"Cameroon\n(n={cm_total})"]
tier_colors = {'GREEN': '#2ECC71', 'ORANGE': '#F39C12', 'RED': '#E74C3C'}

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('white')

bottoms = [0, 0]
for tier in ['GREEN', 'ORANGE', 'RED']:
    vals = [ci_tiers[tier], cm_tiers[tier]]
    bars = ax.bar(x, vals, bottom=bottoms, color=tier_colors[tier], label=tier, width=0.5)
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if v > 0:
            pct = v / [ci_total, cm_total][i] * 100
            ax.text(bar.get_x() + bar.get_width()/2,
                    bottoms[i] + v/2,
                    f'{v}\n({pct:.0f}%)',
                    ha='center', va='center', fontsize=9.5, color='white', fontweight='bold')
    bottoms = [bottoms[i] + vals[i] for i in range(2)]

ax.set_xticks(x)
ax.set_xticklabels(labels_x, fontsize=11)
ax.set_ylabel('Number of vehicles', fontsize=11)
ax.set_title('Price Tier Classification — CI vs CM',
             fontsize=13, fontweight='bold', color=DARK_BLUE)
ax.legend(title='Tier', loc='upper right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor(LIGHT_GREY)
plt.tight_layout()
save(fig, '11_tier_breakdown.png')


In [ ]:
## ── FIGURES 12.x — CI Validation Band Plots ────────────────────────────────
ci_brand_groups = OrderedDict([
    ('Chevrolet · Citroën · Peugeot · Ford',
        ['chevrolet', 'citroen', 'citroën', 'peugeot', 'ford']),
    ('Honda · Hyundai',
        ['honda', 'hyundai']),
    ('Jeep · Land Rover · Mercedes',
        ['jeep', 'landrover', 'mercedesbenz', 'mercedes', 'mercedes-benz']),
    ('Kia · Mazda · Mitsubishi · Nissan',
        ['kia', 'mazda', 'mitsubishi', 'nissan']),
    ('Toyota',
        ['toyota']),
])

for i, (group_label, brands) in enumerate(ci_brand_groups.items(), start=1):
    df_sub = df_validation_ci[df_validation_ci['brand'].isin(brands)].copy()
    if df_sub.empty:
        print(f'  ⚠ Skipping CI group "{group_label}" — no vehicles found')
        continue
    title = f"CI — {group_label} — Customs Price Validation (year ±3)"
    fig = create_validation_plot(df_sub, title, float(df_sub['declared_price'].median()))
    fname = f'12_{i:02d}_validation_ci_{group_label[:20].replace(" ","_").replace("·","").strip("_")}.png'
    save(fig, fname)

print('✓ CI validation plots saved')


In [ ]:
## ── FIGURES 13.x — CM Validation Band Plots ────────────────────────────────
all_cm_brands = sorted(df_validation_cm['brand'].unique())
cm_non_toyota = [b for b in all_cm_brands if b != 'toyota']

cm_brand_groups = OrderedDict()
if cm_non_toyota:
    cm_brand_groups['All Brands (excl. Toyota)'] = cm_non_toyota
if 'toyota' in all_cm_brands:
    cm_brand_groups['Toyota'] = ['toyota']

for i, (group_label, brands) in enumerate(cm_brand_groups.items(), start=1):
    df_sub = df_validation_cm[df_validation_cm['brand'].isin(brands)].copy()
    if df_sub.empty:
        print(f'  ⚠ Skipping CM group "{group_label}" — no vehicles found')
        continue
    title = f"CM — {group_label} — Customs Price Validation (year ±3)"
    fig = create_validation_plot(df_sub, title, float(df_sub['declared_price'].median()))
    fname = f'13_{i:02d}_validation_cm_{group_label[:20].replace(" ","_").replace("·","").strip("_")}.png'
    save(fig, fname)

print('✓ CM validation plots saved')


In [ ]:
## ── Final Manifest ───────────────────────────────────────────────────────────
pngs = sorted(OUT.glob('*.png'))
print(f'\n{"─"*60}')
print(f'  {len(pngs)} figures saved to:  {OUT}')
print(f'{"─"*60}')
for p in pngs:
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<55} {size_kb:>6.0f} KB')
print(f'{"─"*60}')
print('\n✅ Notebook 08 complete — run notebook 09 to build the PPTX.')
